In [1]:
# Install required packages (run in a separate cell/terminal if needed)
# pip install scikit-learn shap tensorflow lime

import os
import time
import copy
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.color import gray2rgb
from skimage.io import imread
from skimage.transform import resize
from skimage.segmentation import mark_boundaries
from skimage import exposure
from scipy.fftpack import dct
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import LinearRegression
from keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications import ResNet50, ResNet101, ResNet152
from tensorflow.keras.applications.resnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Conv2D, MaxPooling2D, Flatten, 
    GlobalAveragePooling2D, BatchNormalization, Activation, Add
)
import shap
from lime import lime_image
from matplotlib.colors import LinearSegmentedColormap


C:\Users\Saurabh\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [ ]:
# Load the models
resnet_50_model = ResNet50(weights='imagenet', include_top=False, pooling='max')
resnet_101_model = ResNet101(weights='imagenet', include_top=False, pooling='max')
resnet_152_model = ResNet152(weights='imagenet', include_top=False, pooling='max')



In [ ]:
def features_of_single_image(I):
    x = tf.expand_dims(I, axis=0)
    x = preprocess_input(x)
    
    # Extract features from each model
    features_50 = resnet_50_model.predict(x, verbose=0)[0]
    features_101 = resnet_101_model.predict(x, verbose=0)[0]
    features_152 = resnet_152_model.predict(x, verbose=0)[0]
    
    # Concatenate the features
    concatenated_features = np.concatenate([features_50, features_101, features_152])
    
    return concatenated_features

In [ ]:
import os
import pandas as pd
from skimage.io import imread

# Directory and labels
image_dir = r"/home/tapti/Desktop/Saurabh/cervical/After Augmentation"
labels = ['0 Normal', '1 Abnormal']

# Create a list to collect the data
data = []
count = 0

# Populate the list with features, labels, image paths, and image name
for label_idx, label in enumerate(labels):
    label_dir = os.path.join(image_dir, label)
    for subdir, dirs, files in os.walk(label_dir):
        for file in files:
            img_path = os.path.join(subdir, file)
            img = imread(img_path)
            features = features_of_single_image(img)  # Extract concatenated features
            image_name = os.path.basename(img_path)  # Get the image name (file name)
            
            # Add the features, label, image name, and image path to the list
            data.append({'features': features, 'labels': label_idx, 'image_paths': img_path, 'image_name': image_name})
            print(count)
            count += 1

# Convert the list of dictionaries to a DataFrame
merged_df = pd.DataFrame(data)


In [ ]:
merged_df

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Separate features and labels
X = merged_df['features'].tolist()
y = merged_df['labels'].tolist()

# Normalize the features using MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Create a new dataframe with normalized features and labels
merged_data_normalized = {'features': X_scaled.tolist(), 'labels': y}
merged_df_normalized = pd.DataFrame(merged_data_normalized)

# Display the merged dataframe with normalized features
print("Merged DataFrame with Normalized Features:")
print(merged_df_normalized)



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from skimage.color import gray2rgb  # In case images are grayscale

# LIME explanation function for images with modified output
def explain_image_with_lime(image, clf):
    explainer = lime.lime_image.LimeImageExplainer()
    explanation = explainer.explain_instance(
        image,
        classifier_fn=lambda img_batch: np.array([clf.predict_proba([features_of_single_image(sample)]) for sample in img_batch])[:, 0, :],
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )
    return explanation



In [ ]:
# Function to plot the original and LIME explanation side-by-side
def plot_original_and_explanation(image, explanation, top_label=0):
    # Get the LIME explanation image
    temp, mask = explanation.get_image_and_mask(
        explanation.top_labels[top_label], positive_only=True, num_features=20, hide_rest=True
    )
    
    # Convert the explanation mask to have only important features, with rest black
    explained_image = np.zeros_like(image)  # Black background
    explained_image[mask != 0] = image[mask != 0]  # Only keep explanation regions

    # Convert grayscale images to RGB for display, if necessary
    if image.ndim == 2 or (image.ndim == 3 and image.shape[2] == 1):  # Grayscale check
        image = gray2rgb(image.squeeze())
        explained_image = gray2rgb(explained_image.squeeze())
    
    # Plot both images
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(image)
    ax[0].set_title("Original Image")
    ax[0].axis('off')
    
    ax[1].imshow(explained_image)
    ax[1].set_title("LIME Explanation")
    ax[1].axis('off')
    
    plt.show()




In [ ]:


# Separate features, labels, and image paths
X = merged_df['features'].tolist()
y = merged_df['labels'].tolist()
image_paths = merged_df['image_paths'].tolist()  # Store image paths for SHAP

num_splits = 5
kernel = 'linear'
print("Combined feature extraction output with 5 fold cross-validation and linear SVM")

kfold = KFold(n_splits=num_splits, shuffle=True, random_state=50)
svm = SVC(kernel=kernel, C=100, probability=True)  # Use probability=True for LIME

accuracy_scores = []
specificity_scores = []
sensitivity_scores = []

for fold, (train_index, test_index) in enumerate(kfold.split(X)):
    print(f"\nFold {fold + 1}:\n")
    
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]
    image_paths_test = np.array(image_paths)[test_index]  # Get image paths for LIME

    # Train SVM model
    svm.fit(X_train, y_train)
    y_pred = svm.predict(X_test)

    # Evaluate the model performance
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp)
    sensitivity = tp / (tp + fn)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    specificity_scores.append(specificity)
    sensitivity_scores.append(sensitivity)
    accuracy_scores.append(accuracy)

    
    # Apply LIME explanations for each test image in this fold
    count = 0
    for image_path in image_paths_test:
        print(f"Explaining image {count + 1}...")
        image = matplotlib.image.imread(image_path)  # Load the image from file using the stored image path
        
        # Generate LIME explanation
        explanation = explain_image_with_lime(image, svm)
        
        # Display the original and explanation images
        plot_original_and_explanation(image, explanation, top_label=0)
        
        count += 1

